# Определение частей речи: SynTagRus, LSTM, GRU и многозадачная модель

В этом варианте задания обучение переведено с небольшого корпуса, размеченного `pymorphy3`, на ручную разметку
**SynTagRus** в формате Universal Dependencies (UPOS). Сравниваются:

- сохранённая базовая CNN;
- новая контекстная BiLSTM;
- новая контекстная BiGRU;
- многозадачная BiLSTM, которая одновременно предсказывает UPOS и 10 морфологических признаков;
- простой ансамбль CNN + LSTM + GRU.

Новые улучшения выделены в описаниях целей ниже. Все корпуса и модели кэшируются, поэтому повторный запуск не обучает
уже сохранённые модели. Для применения без запуска ноутбука используется `predict_pos.py`.

## Кэширование и режим использования готовых моделей

**Что сделано.** Разобранные CoNLL-U-файлы сохраняются в `cache/`, а модели, история обучения и словарь символов —
в `models_syntagrus/`. Эти каталоги добавлены в `.gitignore`, потому что кэш и бинарные модели большие.

`USE_CACHE = True` загружает существующие файлы. Для повторного обучения конкретной модели можно удалить только её
`*.keras` и `*_history.json`; удалять корпусный кэш не нужно. Для быстрого теста кода используется `SMOKE=1`.

После одного обучения можно не выполнять остальные ячейки: запустите в терминале
`python predict_pos.py --text "Мама читает книгу."`. Скрипт загружает только модели и `vocab_syntagrus.json`,
не читает корпуса и не запускает обучение.

In [1]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import re, csv, json, gzip, pickle, random, collections
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

SMOKE = os.environ.get('SMOKE') == '1'
USE_CACHE = True
BASE_DIR = os.getcwd()
DATA_DIR = os.path.abspath(os.path.join(BASE_DIR, '..', 'data'))
CACHE_DIR = os.path.join(BASE_DIR, 'cache_smoke' if SMOKE else 'cache')
MODEL_DIR = os.path.join(BASE_DIR, 'models_smoke' if SMOKE else 'models_syntagrus')
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

SYNTAGRUS_PATH = os.path.join(DATA_DIR, 'ru_syntagrus-ud-2.0-morpho-train.txt')
NERUS_PATH = os.path.join(DATA_DIR, 'nerus_lenta.conllu.gz')
CSV_PATH = os.path.join(DATA_DIR, 'ruscorpora_content.csv')

MAX_WORDS = 40
MAX_CHARS = 20
CHAR_EMB_DIM = 24
N_FILTERS = 48
KERNEL_SIZES = (3, 4, 5)
N_MORPH = 10
EPOCHS = 2 if SMOKE else 5
BATCH_SIZE = 128
PATIENCE = 2
SYNTAGRUS_SENTS = 3000 if SMOKE else None
NERUS_SENTS = 500 if SMOKE else 10000

UPOS = ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON',
        'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']
UPOS2ID = {tag: i for i, tag in enumerate(UPOS)}
PAD_LABEL = -1
PAD_ID, UNK_ID = 0, 1
TOKEN_RE = re.compile(r'\w+(?:-\w+)*|[^\w\s]', re.UNICODE)

print('SMOKE:', SMOKE, '| cache:', CACHE_DIR, '| models:', MODEL_DIR)

SMOKE: False | cache: C:\Users\reino\Desktop\Учеба\Homework\NLP\pos_tagging\cache | models: C:\Users\reino\Desktop\Учеба\Homework\NLP\pos_tagging\models_syntagrus


## Цель 1. Обучение на ручной разметке SynTagRus

**Что сделано**

- Загружен корпус `ru_syntagrus-ud-2.0-morpho-train.txt` (CoNLL-U): 54 529 предложений, около 962 тыс. токенов.
  Части речи размечены вручную, поэтому эталон не зависит от анализатора.
- Набор тегов — 17 UPOS (вместо 12 тегов pymorphy). Появились `PROPN`, `AUX`, `SCONJ`, `DET`, `SYM`.
- Разбиение: 95% предложений — обучение, 5% — валидация. Официальных dev/test в папке нет, поэтому отдельный тест — `nerus_lenta` (цель 2).
- Длинные предложения режутся на куски по `MAX_WORDS = 40` слов, короткие дополняются `PAD`.

**Выгоды.** Обучающих данных примерно в 300 раз больше, чем в прошлой версии (3 200 токенов). Модель учится на
человеческой разметке, а не на ответах анализатора, поэтому её точность — честная оценка.

In [2]:
def read_conllu(path, max_sents=None):
    opener = gzip.open if path.endswith('.gz') else open
    sents, cur = [], []
    with opener(path, 'rt', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line:
                if cur:
                    sents.append(cur)
                    cur = []
                    if max_sents and len(sents) >= max_sents:
                        break
                continue
            if line.startswith('#'):
                continue
            cols = line.split('\t')
            if '-' in cols[0] or '.' in cols[0]:
                continue
            feats = {} if cols[5] == '_' else dict(kv.split('=', 1) for kv in cols[5].split('|'))
            cur.append((cols[1], cols[3], feats))
    if cur and (not max_sents or len(sents) < max_sents):
        sents.append(cur)
    return sents


def cached(name, builder):
    path = os.path.join(CACHE_DIR, name + '.pkl')
    if USE_CACHE and os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    obj = builder()
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    return obj


syntagrus = cached('syntagrus', lambda: read_conllu(SYNTAGRUS_PATH, SYNTAGRUS_SENTS))
nerus = cached('nerus_lenta', lambda: read_conllu(NERUS_PATH, NERUS_SENTS))

random.Random(SEED).shuffle(syntagrus)
n_val = int(0.05 * len(syntagrus))
val_sents, train_sents = syntagrus[:n_val], syntagrus[n_val:]
print(f'SynTagRus: предложений {len(syntagrus)}, обучение {len(train_sents)}, валидация {len(val_sents)}')
print(f'nerus_lenta: предложений {len(nerus)}, токенов {sum(len(s) for s in nerus)}')

SynTagRus: предложений 54529, обучение 51803, валидация 2726
nerus_lenta: предложений 10000, токенов 181663


In [3]:
upos_cnt = collections.Counter(u for s in train_sents for _, u, _ in s)
print('Токенов в обучении:', sum(upos_cnt.values()))
print('Части речи:', dict(upos_cnt.most_common()))

lens = sorted(len(s) for s in train_sents)
print('Длина предложения: медиана', lens[len(lens) // 2], '| 95-й перцентиль', lens[int(0.95 * len(lens))])
word_lens = sorted(len(f) for s in train_sents for f, _, _ in s)
print('Длина слова: 99-й перцентиль', word_lens[int(0.99 * len(word_lens))], '| максимум', word_lens[-1])

Токенов в обучении: 913805
Части речи: {'NOUN': 223821, 'PUNCT': 167644, 'VERB': 108013, 'ADJ': 91530, 'ADP': 84549, 'ADV': 45927, 'PRON': 39282, 'PROPN': 33229, 'CCONJ': 32608, 'PART': 29048, 'DET': 22650, 'SCONJ': 16622, 'NUM': 11579, 'AUX': 6376, 'SYM': 788, 'INTJ': 91, 'X': 48}
Длина предложения: медиана 16 | 95-й перцентиль 37


Длина слова: 99-й перцентиль 15 | максимум 47


In [4]:
def to_chunks(sents):
    return [s[i:i + MAX_WORDS] for s in sents for i in range(0, len(s), MAX_WORDS)]


def forms_of(chunks):
    return [[f for f, _, _ in ch] for ch in chunks]


train_chunks = to_chunks(train_sents)
val_chunks = to_chunks(val_sents)
test_chunks = to_chunks(nerus)

chars = sorted({c for ch in train_chunks for f, _, _ in ch for c in f})
char2id = {c: i + 2 for i, c in enumerate(chars)}
N_CHARS = len(char2id) + 2


def encode_forms(chunks):
    X = np.zeros((len(chunks), MAX_WORDS, MAX_CHARS), dtype='int32')
    for i, ch in enumerate(chunks):
        for j, form in enumerate(ch):
            for k, c in enumerate(form[:MAX_CHARS]):
                X[i, j, k] = char2id.get(c, UNK_ID)
    return X


def encode_targets(chunks, morph_keys=()):
    y = {'upos': np.full((len(chunks), MAX_WORDS), PAD_LABEL, dtype='int32')}
    for k in morph_keys:
        y['morph_' + k] = np.full((len(chunks), MAX_WORDS), PAD_LABEL, dtype='int32')
    for i, ch in enumerate(chunks):
        for j, (_, upos, feats) in enumerate(ch):
            y['upos'][i, j] = UPOS2ID[upos]
            for k in morph_keys:
                if k in feats:
                    y['morph_' + k][i, j] = MORPH_VAL2ID[k].get(feats[k], PAD_LABEL)
    return y


X_train, X_val, X_test = encode_forms(forms_of(train_chunks)), encode_forms(forms_of(val_chunks)), encode_forms(forms_of(test_chunks))
y_train, y_val, y_test = encode_targets(train_chunks), encode_targets(val_chunks), encode_targets(test_chunks)
print('X_train:', X_train.shape, '| символьный словарь:', N_CHARS)

X_train: (53606, 40, 20) | символьный словарь: 161


### Базовая CNN, LSTM и GRU

**Что добавлено для улучшения.**

1. Символы каждого слова сначала превращаются в вектор через небольшую CNN с ядрами 3, 4 и 5. Это сохраняет
   требование посимвольной обработки, но существенно быстрее, чем LSTM по каждому символу.
2. Контекст предложения обрабатывается двумя отдельными моделями: `Bidirectional(LSTM)` и `Bidirectional(GRU)`.
   Поэтому сравнивается не только одна рекуррентная сеть, а две распространённые ячейки памяти.
3. Их представление объединяется с локальной свёрткой по словам. Так модель видит и соседние слова, и контекст
   всего предложения.
4. Для редких частей речи используется мягкое взвешивание классов в функции потерь. Это попытка улучшить не только accuracy,
   но и macro-F1; обе метрики считаются отдельно, чтобы частые существительные и знаки препинания не скрывали редкие классы.

Старую CNN оставляем как baseline. Новые модели сохраняются отдельными файлами `lstm_single.keras` и
`gru_single.keras`, поэтому их можно сравнивать и загружать независимо.

In [5]:
def masked_loss(y_true, y_pred):
    mask = tf.cast(y_true != PAD_LABEL, tf.float32)
    ce = keras.losses.sparse_categorical_crossentropy(tf.maximum(y_true, 0), y_pred)
    return tf.reduce_sum(ce * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)


def masked_accuracy(y_true, y_pred):
    mask = tf.cast(y_true != PAD_LABEL, tf.float32)
    hit = tf.cast(tf.equal(tf.maximum(y_true, 0), tf.argmax(y_pred, axis=-1, output_type=tf.int32)), tf.float32)
    return tf.reduce_sum(hit * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)


tag_counts = np.bincount(y_train['upos'][y_train['upos'] != PAD_LABEL], minlength=len(UPOS)).astype('float32')
nonzero = tag_counts[tag_counts > 0]
UPOS_WEIGHTS = np.clip(np.sqrt(np.median(nonzero) / np.maximum(tag_counts, 1.0)), 0.75, 3.0).astype('float32')
UPOS_WEIGHTS = tf.constant(UPOS_WEIGHTS)


def weighted_upos_loss(y_true, y_pred):
    mask = tf.cast(y_true != PAD_LABEL, tf.float32)
    labels = tf.cast(tf.maximum(y_true, 0), tf.int32)
    ce = keras.losses.sparse_categorical_crossentropy(labels, y_pred)
    weights = tf.gather(UPOS_WEIGHTS, labels)
    return tf.reduce_sum(ce * weights * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)


def build_baseline_cnn():
    # Архитектура старой CNN — baseline и загрузка старого файла.
    char_in = keras.Input(shape=(MAX_CHARS,), dtype='int32')
    x = layers.Embedding(N_CHARS, 32)(char_in)
    pooled = [layers.GlobalMaxPooling1D()(layers.Conv1D(64, k, activation='relu', padding='same')(x))
              for k in KERNEL_SIZES]
    word_vec = layers.Dropout(0.3)(layers.Concatenate()(pooled))
    word_encoder = keras.Model(char_in, word_vec, name='baseline_char_cnn')
    sent_in = keras.Input(shape=(MAX_WORDS, MAX_CHARS), dtype='int32')
    h = layers.TimeDistributed(word_encoder)(sent_in)
    h = layers.Conv1D(128, 3, activation='relu', padding='same')(h)
    h = layers.TimeDistributed(layers.Dense(64, activation='relu'))(h)
    h = layers.Dropout(0.3)(h)
    out = {'upos': layers.TimeDistributed(layers.Dense(len(UPOS), activation='softmax'), name='upos')(h)}
    return keras.Model(sent_in, out, name='cnn_baseline')


def build_context_model(kind, morph=False):
    # Улучшенная модель: char-CNN для слова + LSTM/GRU для контекста предложения.
    char_in = keras.Input(shape=(MAX_CHARS,), dtype='int32')
    x = layers.Embedding(N_CHARS, CHAR_EMB_DIM)(char_in)
    pooled = [layers.GlobalMaxPooling1D()(layers.Conv1D(N_FILTERS, k, activation='relu', padding='same')(x))
              for k in KERNEL_SIZES]
    word_vec = layers.Dense(160, activation='relu')(layers.Concatenate()(pooled))
    word_encoder = keras.Model(char_in, word_vec, name='char_cnn_word_encoder')
    sent_in = keras.Input(shape=(MAX_WORDS, MAX_CHARS), dtype='int32')
    words = layers.TimeDistributed(word_encoder)(sent_in)
    local = layers.Conv1D(96, 3, activation='relu', padding='same')(words)
    if kind == 'lstm':
        context = layers.Bidirectional(layers.LSTM(96, return_sequences=True, dropout=0.15))(words)
    elif kind == 'gru':
        context = layers.Bidirectional(layers.GRU(96, return_sequences=True, dropout=0.15))(words)
    else:
        raise ValueError('kind должен быть lstm или gru')
    h = layers.Concatenate()([words, local, context])
    h = layers.LayerNormalization()(h)
    h = layers.TimeDistributed(layers.Dense(128, activation='relu'))(h)
    h = layers.Dropout(0.25)(h)
    outputs = {'upos': layers.TimeDistributed(layers.Dense(len(UPOS), activation='softmax'), name='upos')(h)}
    if morph:
        for key in MORPH_KEYS:
            name = 'morph_' + key
            outputs[name] = layers.TimeDistributed(
                layers.Dense(len(MORPH_VAL2ID[key]), activation='softmax'), name=name)(h)
    return keras.Model(sent_in, outputs, name=kind + ('_multi' if morph else '_single'))


def predict_dict(model, X):
    output = model.predict(X, batch_size=256, verbose=0)
    if isinstance(output, dict):
        return output
    if isinstance(output, list):
        return dict(zip(model.output_names, output))
    return {'upos': output}


def compile_and_train(name, model, y_tr, y_va):
    path = os.path.join(MODEL_DIR, name + '.keras')
    history_path = os.path.join(MODEL_DIR, name + '_history.json')
    if USE_CACHE and os.path.exists(path) and os.path.exists(history_path):
        print(name, ': загружена из кэша')
        with open(history_path, encoding='utf-8') as f:
            history = json.load(f)
        return keras.models.load_model(path, compile=False), history
    losses = {output_name: (weighted_upos_loss if output_name == 'upos' else masked_loss)
              for output_name in model.output_names}
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=losses,
                  loss_weights={n: (1.0 if n == 'upos' else 0.2) for n in model.output_names},
                  metrics={'upos': [masked_accuracy]})
    stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                         restore_best_weights=True, verbose=1)
    history = model.fit(X_train, y_tr, validation_data=(X_val, y_va), epochs=EPOCHS,
                        batch_size=BATCH_SIZE, callbacks=[stop], verbose=2)
    model.save(path)
    history_dict = {key: [float(value) for value in values] for key, values in history.history.items()}
    with open(history_path, 'w', encoding='utf-8') as f:
        json.dump(history_dict, f)
    return model, history_dict


def make_and_train(name, kind, morph=False):
    model = build_baseline_cnn() if kind == 'cnn' else build_context_model(kind, morph)
    train_targets = y_train_m if morph else y_train
    val_targets = y_val_m if morph else y_val
    targets = {key: train_targets[key] for key in model.output_names}
    val_targets = {key: val_targets[key] for key in model.output_names}
    return compile_and_train(name, model, targets, val_targets)


cnn_single, hist_cnn = make_and_train('cnn_single', 'cnn')

cnn_single : загружена из кэша


In [6]:
lstm_single, hist_lstm = make_and_train('lstm_single', 'lstm')

Epoch 1/5


419/419 - 57s - 136ms/step - loss: 0.3614 - masked_accuracy: 0.8702 - val_loss: 0.1381 - val_masked_accuracy: 0.9471


Epoch 2/5


419/419 - 47s - 113ms/step - loss: 0.1216 - masked_accuracy: 0.9542 - val_loss: 0.1048 - val_masked_accuracy: 0.9604


Epoch 3/5


419/419 - 48s - 114ms/step - loss: 0.0962 - masked_accuracy: 0.9636 - val_loss: 0.0898 - val_masked_accuracy: 0.9673


Epoch 4/5


419/419 - 42s - 100ms/step - loss: 0.0835 - masked_accuracy: 0.9683 - val_loss: 0.0826 - val_masked_accuracy: 0.9690


Epoch 5/5


419/419 - 40s - 96ms/step - loss: 0.0752 - masked_accuracy: 0.9711 - val_loss: 0.0803 - val_masked_accuracy: 0.9709


Restoring model weights from the end of the best epoch: 5.


### Отдельная модель GRU

GRU содержит меньше управляющих ворот, чем LSTM, поэтому обычно обучается быстрее и иногда лучше переносится на
короткие предложения. Она обучается отдельно, сохраняется в `gru_single.keras` и сравнивается с CNN и LSTM на одном
и том же `nerus_lenta`.

In [7]:
gru_single, hist_gru = make_and_train('gru_single', 'gru')

Epoch 1/5


419/419 - 62s - 149ms/step - loss: 0.3611 - masked_accuracy: 0.8699 - val_loss: 0.1429 - val_masked_accuracy: 0.9443


Epoch 2/5


419/419 - 39s - 92ms/step - loss: 0.1214 - masked_accuracy: 0.9546 - val_loss: 0.0975 - val_masked_accuracy: 0.9634


Epoch 3/5


419/419 - 39s - 92ms/step - loss: 0.0958 - masked_accuracy: 0.9640 - val_loss: 0.0856 - val_masked_accuracy: 0.9681


Epoch 4/5


419/419 - 38s - 91ms/step - loss: 0.0829 - masked_accuracy: 0.9684 - val_loss: 0.0799 - val_masked_accuracy: 0.9700


Epoch 5/5


419/419 - 38s - 91ms/step - loss: 0.0751 - masked_accuracy: 0.9713 - val_loss: 0.0759 - val_masked_accuracy: 0.9716


Restoring model weights from the end of the best epoch: 5.


## Цель 2. Оценка на nerus_lenta и ансамбль

`nerus_lenta` — новости Lenta.ru в формате CoNLL-U. В отличие от SynTagRus, этот корпус не используется для обучения,
поэтому он показывает переносимость на другой источник и стиль. Для эксперимента берутся первые 10 000 предложений
(около 180 тыс. токенов), а чтение всего 2-гигабайтного архива не требуется.

Кроме отдельных CNN, LSTM и GRU добавлен простой ансамбль: вероятности трёх моделей усредняются. Веса выбираются
только по validation SynTagRus, а итоговая оценка считается на nerus_lenta. Проверяется, уменьшает ли это ошибки, характерные
для одной архитектуры; на полном тесте результат сохраняется для сравнения.

На полном тесте лучшей оказалась baseline CNN; это зафиксировано в итоговой таблице, поэтому ансамбль не выдаётся за улучшение.


In [8]:
def evaluate_probabilities(name, probabilities, target):
    prediction = probabilities.argmax(-1)
    mask = target != PAD_LABEL
    report = classification_report(target[mask], prediction[mask], labels=range(len(UPOS)),
                                   target_names=UPOS, zero_division=0, output_dict=True)
    results[name] = prediction
    reports[name] = report
    return {'accuracy': float((prediction[mask] == target[mask]).mean()),
            'macro F1': report['macro avg']['f1-score'],
            'weighted F1': report['weighted avg']['f1-score']}


def upos_probabilities(model, X):
    return predict_dict(model, X)['upos']


def evaluate_upos(name, model):
    probabilities = upos_probabilities(model, X_test)
    probabilities_test[name] = probabilities
    return evaluate_probabilities(name, probabilities, y_test['upos'])


results, reports, probabilities_test = {}, {}, {}
probabilities_val = {
    'CNN': upos_probabilities(cnn_single, X_val),
    'LSTM': upos_probabilities(lstm_single, X_val),
    'GRU': upos_probabilities(gru_single, X_val),
}
for name, model in [('CNN', cnn_single), ('LSTM', lstm_single), ('GRU', gru_single)]:
    probabilities_test[name] = upos_probabilities(model, X_test)

scores = {name: evaluate_probabilities(name, probabilities_test[name], y_test['upos'])
          for name in probabilities_test}

candidate_weights = [(0.5, 0.25, 0.25), (0.4, 0.3, 0.3), (0.25, 0.5, 0.25), (0.25, 0.25, 0.5)]
best_weights, best_val_f1 = candidate_weights[0], -1.0
for weights in candidate_weights:
    val_mix = sum(weight * probabilities_val[name] for weight, name in zip(weights, ('CNN', 'LSTM', 'GRU')))
    mask = y_val['upos'] != PAD_LABEL
    val_report = classification_report(y_val['upos'][mask], val_mix.argmax(-1)[mask],
                                       labels=range(len(UPOS)), target_names=UPOS,
                                       zero_division=0, output_dict=True)
    if val_report['macro avg']['f1-score'] > best_val_f1:
        best_val_f1 = val_report['macro avg']['f1-score']
        best_weights = weights
ensemble_test = sum(weight * probabilities_test[name] for weight, name in zip(best_weights, ('CNN', 'LSTM', 'GRU')))
probabilities_test['Ансамбль CNN+LSTM+GRU'] = ensemble_test
scores['Ансамбль CNN+LSTM+GRU'] = evaluate_probabilities('Ансамбль CNN+LSTM+GRU', ensemble_test, y_test['upos'])
print('Веса ансамбля, выбранные по validation:', best_weights)
pd.DataFrame(scores).T.round(4)

Веса ансамбля, выбранные по validation: (0.25, 0.25, 0.5)


,accuracy,macro F1,weighted F1
CNN,0.9566,0.8317,0.9531
LSTM,0.9255,0.7977,0.9233
GRU,0.9202,0.7844,0.9315
Ансамбль CNN+LSTM+GRU,0.9305,0.7856,0.9365


In [9]:
pd.DataFrame(reports['CNN']).T.loc[UPOS].round(2)

,precision,recall,f1-score,support
ADJ,0.96,0.96,0.96,16177.0
ADP,0.94,1.00,0.97,20667.0
ADV,0.94,0.90,0.92,4611.0
AUX,0.96,0.86,0.91,1204.0
CCONJ,0.98,0.99,0.98,4255.0
DET,0.89,0.94,0.92,2761.0
INTJ,0.00,0.00,0.00,11.0
NOUN,0.98,0.97,0.97,49929.0
NUM,0.94,0.97,0.96,3460.0
PART,0.90,0.94,0.92,2150.0


## Цель 3. Морфология: 10 признаков и многозадачная модель

**Выбор признаков.** В SynTagRus около 40 морфологических признаков. Берём 10 самых частых
(по числу токенов, у которых признак указан), чтобы не перегружать модель:

In [10]:
key_counts = collections.Counter(k for s in train_sents for _, _, f in s for k in f)
print('Признаки по частоте:', key_counts.most_common(15))

MORPH_KEYS = [k for k, _ in key_counts.most_common(N_MORPH)]
MORPH_VAL2ID = {}
for k in MORPH_KEYS:
    vals = sorted({f[k] for s in train_sents for _, _, f in s if k in f})
    MORPH_VAL2ID[k] = {v: i for i, v in enumerate(vals)}
print({k: len(v) for k, v in MORPH_VAL2ID.items()})

Признаки по частоте: [('Number', 490122), ('Case', 420404), ('Gender', 390099), ('Animacy', 282244), ('Degree', 141679), ('Aspect', 109937), ('VerbForm', 109937), ('Voice', 90048), ('Tense', 89224), ('Mood', 71391), ('Person', 54572), ('Variant', 9710), ('Reflex', 1324)]


{'Number': 2, 'Case': 8, 'Gender': 3, 'Animacy': 2, 'Degree': 3, 'Aspect': 2, 'VerbForm': 4, 'Voice': 2, 'Tense': 3, 'Mood': 2}


**Многозадачная модель.** Та же контекстная LSTM, но поверх общего представления предложения стоит 11 выходов:
UPOS и по одному softmax на каждый из 10 признаков. Потери складываются с весами: UPOS — 1.0, каждый признак — 0.2.
Если признака у слова нет (в UD он `_`), метка `PAD_LABEL` не учитывается в потерях. Эта модель нужна для сравнения
с одной задачей: проверяем, помогает ли морфология части речи.

In [11]:
y_train_m = encode_targets(train_chunks, MORPH_KEYS)
y_val_m = encode_targets(val_chunks, MORPH_KEYS)
y_test_m = encode_targets(test_chunks, MORPH_KEYS)

# Та же контекстная LSTM, но с дополнительными выходами морфологии.
lstm_multi, hist_multi = make_and_train('lstm_multi', 'lstm', morph=True)
multi_probabilities = predict_dict(lstm_multi, X_test)
scores['Многозадачная LSTM'] = evaluate_probabilities('Многозадачная LSTM', multi_probabilities['upos'], y_test['upos'])
pd.DataFrame(scores).T.round(4)

Epoch 1/5


419/419 - 84s - 201ms/step - loss: 1.1098 - morph_Animacy_loss: 0.3429 - morph_Aspect_loss: 0.3438 - morph_Case_loss: 0.9705 - morph_Degree_loss: 0.0830 - morph_Gender_loss: 0.6381 - morph_Mood_loss: 0.0391 - morph_Number_loss: 0.3242 - morph_Tense_loss: 0.1986 - morph_VerbForm_loss: 0.1660 - morph_Voice_loss: 0.1648 - upos_loss: 0.4553 - upos_masked_accuracy: 0.8394 - val_loss: 0.4891 - val_morph_Animacy_loss: 0.2503 - val_morph_Aspect_loss: 0.1759 - val_morph_Case_loss: 0.5037 - val_morph_Degree_loss: 0.0256 - val_morph_Gender_loss: 0.3304 - val_morph_Mood_loss: 0.0140 - val_morph_Number_loss: 0.1570 - val_morph_Tense_loss: 0.0547 - val_morph_VerbForm_loss: 0.0171 - val_morph_Voice_loss: 0.0902 - val_upos_loss: 0.1653 - val_upos_masked_accuracy: 0.9390


Epoch 2/5


419/419 - 63s - 150ms/step - loss: 0.4826 - morph_Animacy_loss: 0.2299 - morph_Aspect_loss: 0.1700 - morph_Case_loss: 0.5026 - morph_Degree_loss: 0.0242 - morph_Gender_loss: 0.3354 - morph_Mood_loss: 0.0123 - morph_Number_loss: 0.1459 - morph_Tense_loss: 0.0586 - morph_VerbForm_loss: 0.0275 - morph_Voice_loss: 0.0859 - upos_loss: 0.1641 - upos_masked_accuracy: 0.9402 - val_loss: 0.3671 - val_morph_Animacy_loss: 0.2037 - val_morph_Aspect_loss: 0.1208 - val_morph_Case_loss: 0.3895 - val_morph_Degree_loss: 0.0164 - val_morph_Gender_loss: 0.2502 - val_morph_Mood_loss: 0.0064 - val_morph_Number_loss: 0.1239 - val_morph_Tense_loss: 0.0375 - val_morph_VerbForm_loss: 0.0090 - val_morph_Voice_loss: 0.0617 - val_upos_loss: 0.1233 - val_upos_masked_accuracy: 0.9547


Epoch 3/5


419/419 - 95s - 226ms/step - loss: 0.3852 - morph_Animacy_loss: 0.1963 - morph_Aspect_loss: 0.1285 - morph_Case_loss: 0.4144 - morph_Degree_loss: 0.0158 - morph_Gender_loss: 0.2676 - morph_Mood_loss: 0.0089 - morph_Number_loss: 0.1118 - morph_Tense_loss: 0.0420 - morph_VerbForm_loss: 0.0191 - morph_Voice_loss: 0.0739 - upos_loss: 0.1296 - upos_masked_accuracy: 0.9520 - val_loss: 0.3109 - val_morph_Animacy_loss: 0.1722 - val_morph_Aspect_loss: 0.1049 - val_morph_Case_loss: 0.3399 - val_morph_Degree_loss: 0.0117 - val_morph_Gender_loss: 0.2096 - val_morph_Mood_loss: 0.0046 - val_morph_Number_loss: 0.0828 - val_morph_Tense_loss: 0.0339 - val_morph_VerbForm_loss: 0.0052 - val_morph_Voice_loss: 0.0540 - val_upos_loss: 0.1072 - val_upos_masked_accuracy: 0.9610


... (2 outputs left out)

Epoch 5/5


419/419 - 60s - 144ms/step - loss: 0.3052 - morph_Animacy_loss: 0.1618 - morph_Aspect_loss: 0.0949 - morph_Case_loss: 0.3453 - morph_Degree_loss: 0.0105 - morph_Gender_loss: 0.2088 - morph_Mood_loss: 0.0060 - morph_Number_loss: 0.0847 - morph_Tense_loss: 0.0286 - morph_VerbForm_loss: 0.0116 - morph_Voice_loss: 0.0623 - upos_loss: 0.1023 - upos_masked_accuracy: 0.9614 - val_loss: 0.2649 - val_morph_Animacy_loss: 0.1463 - val_morph_Aspect_loss: 0.0807 - val_morph_Case_loss: 0.2963 - val_morph_Degree_loss: 0.0099 - val_morph_Gender_loss: 0.1745 - val_morph_Mood_loss: 0.0033 - val_morph_Number_loss: 0.0673 - val_morph_Tense_loss: 0.0235 - val_morph_VerbForm_loss: 0.0027 - val_morph_Voice_loss: 0.0517 - val_upos_loss: 0.0936 - val_upos_masked_accuracy: 0.9652


Restoring model weights from the end of the best epoch: 5.


,accuracy,macro F1,weighted F1
CNN,0.9566,0.8317,0.9531
LSTM,0.9255,0.7977,0.9233
GRU,0.9202,0.7844,0.9315
Ансамбль CNN+LSTM+GRU,0.9305,0.7856,0.9365
Многозадачная LSTM,0.9145,0.7795,0.9145


In [12]:
pred_m = predict_dict(lstm_multi, X_test)
morph_rows = []
for k in MORPH_KEYS:
    name = 'morph_' + k
    mask = y_test_m[name] != PAD_LABEL
    acc = (pred_m[name].argmax(-1)[mask] == y_test_m[name][mask]).mean()
    morph_rows.append({'признак': k, 'токенов с меткой': int(mask.sum()), 'точность': round(float(acc), 3)})
pd.DataFrame(morph_rows)

,признак,токенов с меткой,точность
0,Number,103423,0.977
1,Case,87254,0.890
2,Gender,87852,0.923
3,Animacy,66697,0.920
4,Degree,19325,0.990
5,Aspect,21978,0.976
6,VerbForm,21978,0.999
7,Voice,19195,0.965
8,Tense,19006,0.979
9,Mood,15587,0.999


In [13]:
vocab = {'char2id': char2id, 'upos': UPOS, 'max_words': MAX_WORDS, 'max_chars': MAX_CHARS,
         'morph_keys': MORPH_KEYS, 'morph_values': MORPH_VAL2ID}
for destination in (os.path.join(BASE_DIR, 'vocab_syntagrus.json'),
                     os.path.join(MODEL_DIR, 'vocab.json')):
    with open(destination, 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False, indent=1)
print('Словарь сохранён в', os.path.join(BASE_DIR, 'vocab_syntagrus.json'))

Словарь сохранён в C:\Users\reino\Desktop\Учеба\Homework\NLP\pos_tagging\vocab_syntagrus.json


### Функция разбора

Она нужна в целях 4 и 5. Для текста разбивает слова, прогоняет его через три модели и собирает таблицу.

In [14]:
id2val = {key: {i: value for value, i in MORPH_VAL2ID[key].items()} for key in MORPH_KEYS}


def analyze(text):
    # Разбор в текущей сессии; для полностью отдельного запуска есть predict_pos.py.
    tokens = TOKEN_RE.findall(text)
    rows = []
    for start in range(0, len(tokens), MAX_WORDS):
        chunk = tokens[start:start + MAX_WORDS]
        X = encode_forms([chunk])
        p_cnn = upos_probabilities(cnn_single, X)[0]
        p_lstm = upos_probabilities(lstm_single, X)[0]
        p_gru = upos_probabilities(gru_single, X)[0]
        p_ensemble = sum(weight * p for weight, p in zip(best_weights, (p_cnn, p_lstm, p_gru)))
        p_multi = predict_dict(lstm_multi, X)
        for j, word in enumerate(chunk):
            multi_tag = UPOS[int(p_multi['upos'][0, j].argmax())]
            features = []
            for key in MORPH_KEYS:
                probabilities = p_multi['morph_' + key][0, j]
                if probabilities.max() >= 0.5:
                    features.append(f'{key}={id2val[key][int(probabilities.argmax())]}')
            rows.append({'слово': word,
                         'CNN': UPOS[int(p_cnn[j].argmax())],
                         'LSTM': UPOS[int(p_lstm[j].argmax())],
                         'GRU': UPOS[int(p_gru[j].argmax())],
                         'ансамбль': UPOS[int(p_ensemble[j].argmax())],
                         'уверенность ансамбля': round(float(p_ensemble[j].max()), 3),
                         'многозадачная LSTM': multi_tag,
                         'признаки': '|'.join(features)})
    return pd.DataFrame(rows)

## Цель 4. Проверка на омонимичных словах

**Что считаем.** Омонимичное слово — форма, которая в SynTagRus встречается с разными частями речи. Берём формы
с не менее чем 50 вхождениями, у которых минимальная доля второй части речи не меньше 15%
(например, «что» — PRON или SCONJ, «как» — ADV или SCONJ). Проверяем точность на этих словах в `nerus_lenta`.

Затем берём примеры из `ruscorpora_content.csv`, где омонимия снята вручную (колонка `Ambiguity`). Метки частей речи в этом файле
нет, поэтому проверка качественная: смотрим, какой тег модель даёт центральному слову.

In [15]:
upos_by_form = collections.defaultdict(collections.Counter)
for s in train_sents:
    for f, u, _ in s:
        upos_by_form[f.lower()][u] += 1

AMB = {f for f, c in upos_by_form.items()
       if sum(c.values()) >= 50 and len(c) >= 2 and min(c.values()) / sum(c.values()) >= 0.15}
top_amb = sorted(AMB, key=lambda f: -sum(upos_by_form[f].values()))[:10]
print('Омонимичных форм:', len(AMB))

flat = [(i, j, f.lower()) for i, ch in enumerate(test_chunks) for j, (f, _, _) in enumerate(ch)]
amb_pos = np.zeros(y_test['upos'].shape, dtype=bool)
for i, j, f in flat:
    if f in AMB:
        amb_pos[i, j] = True

rows = []
for form in top_amb:
    pos = [(i, j) for i, j, f in flat if f == form]
    if not pos:
        continue
    ii, jj = zip(*pos)
    row = {'форма': form,
           'разборы в train': ', '.join(f'{u}:{n}' for u, n in upos_by_form[form].most_common(3)),
           'токенов в nerus': len(pos)}
    for name, pred in results.items():
        row[name] = round(float((pred[ii, jj] == y_test['upos'][ii, jj]).mean()), 3)
    rows.append(row)
pd.DataFrame(rows)

Омонимичных форм: 84


,форма,разборы в train,токенов в nerus,CNN,LSTM,GRU,Ансамбль CNN+LSTM+GRU,Многозадачная LSTM
0,его,"DET:1698, PRON:918",622,0.791,0.791,0.780,0.793,0.796
1,их,"DET:1083, PRON:807",245,0.796,0.763,0.743,0.755,0.780
2,ее,"DET:717, PRON:519",282,0.823,0.801,0.823,0.830,0.819
3,будет,"AUX:929, VERB:180",236,0.903,0.924,0.907,0.911,0.911
4,этом,"PRON:689, DET:416",685,0.990,0.953,0.920,0.956,0.982
5,более,"ADV:802, NUM:203",186,0.645,0.710,0.511,0.581,0.484
6,быть,"AUX:612, VERB:319",74,0.892,0.905,0.905,0.919,0.919
7,этого,"PRON:470, DET:416",140,0.921,0.907,0.921,0.907,0.964
8,нет,"VERB:592, PART:141",76,0.961,0.934,0.947,0.961,0.934
9,всех,"DET:440, PRON:126",62,0.952,0.919,0.919,0.919,0.968


In [16]:
summary = {}
for name, pred in results.items():
    summary[name] = {'все слова': round(float((pred[y_test['upos'] != PAD_LABEL] == y_test['upos'][y_test['upos'] != PAD_LABEL]).mean()), 3),
                     'омонимы': round(float((pred[amb_pos] == y_test['upos'][amb_pos]).mean()), 3)}
pd.DataFrame(summary).T

,все слова,омонимы
CNN,0.957,0.869
LSTM,0.925,0.842
GRU,0.920,0.844
Ансамбль CNN+LSTM+GRU,0.931,0.858
Многозадачная LSTM,0.914,0.857


In [17]:
with open(CSV_PATH, encoding='utf-8', newline='') as f:
    csv_rows = list(csv.DictReader(f, delimiter=';'))
manual = [r for r in csv_rows if r['Ambiguity'] == 'омонимия снята вручную' and r['Center'].lower() in AMB]
print('Примеров "омонимия снята вручную" с омонимичным словом:', len(manual))

Примеров "омонимия снята вручную" с омонимичным словом: 0


In [18]:
examples = []
for row in manual[:8]:
    table = analyze(row['Full context'])
    hit = table[table['слово'].str.lower() == row['Center'].lower()]
    if len(hit):
        examples.append({'центральное слово': row['Center'],
                         'CNN': hit.iloc[0]['CNN'],
                         'LSTM': hit.iloc[0]['LSTM'],
                         'GRU': hit.iloc[0]['GRU'],
                         'ансамбль': hit.iloc[0]['ансамбль'],
                         'контекст': row['Full context'][:90]})
pd.DataFrame(examples)

""


## Цель 5. Разбор слова или предложения

В переменную `text` можно записать одно слово или предложение и выполнить ячейку `analyze(text)`. Для каждого слова
показываются предсказания CNN, LSTM, GRU, ансамбля и многозадачной LSTM, а также уверенность ансамбля и найденные
морфологические признаки.

Для запуска **без обучения и без загрузки корпусов** используйте отдельный `predict_pos.py`:

```bash
python predict_pos.py --text "Что делать, если пароль забыт?"
```

In [19]:
text = 'Вице-премьер по социальным вопросам вчера посетил больницу в Петербурге.'
analyze(text)

,слово,CNN,LSTM,GRU,ансамбль,уверенность ансамбля,многозадачная LSTM,признаки
0,Вице-премьер,NOUN,NOUN,NOUN,NOUN,0.988,NOUN,Number=Sing|Case=Nom|Gender=Fem|Animacy=Anim|D...
1,по,ADP,ADP,ADP,ADP,1.000,ADP,Number=Sing|Case=Acc|Animacy=Inan|Degree=Pos|A...
2,социальным,ADJ,ADJ,ADJ,ADJ,1.000,ADJ,Number=Plur|Case=Dat|Gender=Masc|Animacy=Inan|...
3,вопросам,NOUN,NOUN,NOUN,NOUN,1.000,NOUN,Number=Plur|Case=Dat|Gender=Masc|Animacy=Inan|...
4,вчера,ADV,ADV,ADV,ADV,0.998,ADV,Number=Sing|Case=Gen|Gender=Masc|Animacy=Anim|...
5,посетил,VERB,VERB,VERB,VERB,1.000,VERB,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...
6,больницу,NOUN,NOUN,NOUN,NOUN,1.000,NOUN,Number=Sing|Case=Acc|Gender=Fem|Animacy=Inan|D...
7,в,ADP,ADP,ADP,ADP,1.000,ADP,Number=Plur|Case=Acc|Gender=Fem|Animacy=Inan|D...
8,Петербурге,PROPN,PROPN,PROPN,PROPN,0.996,PROPN,Number=Sing|Case=Loc|Gender=Masc|Animacy=Inan|...
9,.,PUNCT,PUNCT,PUNCT,PUNCT,1.000,PUNCT,Number=Sing|Case=Acc|Gender=Masc|Animacy=Inan|...


In [20]:
analyze('Мама')

,слово,CNN,LSTM,GRU,ансамбль,уверенность ансамбля,многозадачная LSTM,признаки
0,Мама,NOUN,NOUN,NOUN,NOUN,0.862,NOUN,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...


In [21]:
text = 'Что делать, если пароль забыт?'
analyze(text)

,слово,CNN,LSTM,GRU,ансамбль,уверенность ансамбля,многозадачная LSTM,признаки
0,Что,PRON,PRON,PRON,PRON,0.996,PRON,Number=Sing|Case=Acc|Gender=Neut|Animacy=Inan|...
1,делать,VERB,VERB,VERB,VERB,1.000,VERB,Number=Sing|Case=Nom|Gender=Fem|Animacy=Inan|D...
2,",",PUNCT,PUNCT,PUNCT,PUNCT,1.000,PUNCT,Number=Sing|Case=Acc|Gender=Masc|Animacy=Inan|...
3,если,SCONJ,SCONJ,SCONJ,SCONJ,0.999,SCONJ,Number=Sing|Case=Acc|Gender=Fem|Animacy=Inan|D...
4,пароль,NOUN,NOUN,NOUN,NOUN,0.999,NOUN,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...
5,забыт,VERB,NOUN,VERB,VERB,0.706,NOUN,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...
6,?,PUNCT,PUNCT,PUNCT,PUNCT,1.000,PUNCT,Number=Sing|Case=Acc|Gender=Masc|Animacy=Inan|...


## Запуск только готовых моделей

Эта ячейка демонстрирует режим inference-only. После обучения ноутбука можно закрыть ядро и использовать тот же код
через `predict_pos.py`: загружаются только сохранённые `*.keras` и небольшой `vocab_syntagrus.json`.

In [22]:
from predict_pos import tag_text

text = 'Что делать, если пароль забыт?'
tag_text(text)

,слово,CNN,уверенность CNN,LSTM,уверенность LSTM,GRU,уверенность GRU,Многозадачная LSTM,уверенность Многозадачная LSTM,признаки (многозадачная модель)
0,Что,PRON,0.993,PRON,0.999,PRON,0.996,PRON,0.996,Number=Sing|Case=Acc|Gender=Neut|Animacy=Inan|...
1,делать,VERB,1.000,VERB,1.000,VERB,1.000,VERB,0.999,Number=Sing|Case=Nom|Gender=Fem|Animacy=Inan|D...
2,",",PUNCT,1.000,PUNCT,1.000,PUNCT,1.000,PUNCT,1.000,Number=Sing|Case=Acc|Gender=Masc|Animacy=Inan|...
3,если,SCONJ,1.000,SCONJ,1.000,SCONJ,0.999,SCONJ,1.000,Number=Sing|Case=Acc|Gender=Fem|Animacy=Inan|D...
4,пароль,NOUN,1.000,NOUN,1.000,NOUN,0.999,NOUN,0.984,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...
5,забыт,VERB,0.832,NOUN,0.657,VERB,0.834,NOUN,0.642,Number=Sing|Case=Nom|Gender=Masc|Animacy=Inan|...
6,?,PUNCT,1.000,PUNCT,1.000,PUNCT,1.000,PUNCT,1.000,Number=Sing|Case=Acc|Gender=Masc|Animacy=Inan|...
